<a href="https://colab.research.google.com/github/Ajay07pandey/Gen_AI_Projects/blob/main/Multi_model_AI_Doctor/Multi_model_AI_ChatBot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Brain of the Doctor Multimodel

In [ ]:
!pip install -q langchain_groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.0/136.0 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.5/471.5 kB 22.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 0.3.27 requires langchain-core<1.0.0,>=0.3.72, but you have langchain-core 1.0.5 which is incompatible.


In [ ]:
import os
import getpass
if "GROQ_API_KEY" not in os.environ:
  os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key")

Enter your Groq API Key··········


In [ ]:
# Convert Image to required format
import base64
image_path = "/content/Acne.png"
image_file = open(image_path, "rb")
encoded_image = base64.b64encode(image_file.read()).decode('utf-8')

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
llm = ChatGroq (
    model = "meta-llama/llama-4-scout-17b-16e-instruct"
)


In [ ]:
query = "Check my face"
# Correct multimodal message format for LangChain
message = HumanMessage(
    content=[
        {"type": "text", "text": query},
        {
            "type": "image_url",
            "image_url": {
                "url": f"data:image/jpeg;base64,{encoded_image}"
            }
        }
    ]
)


In [ ]:
# Must pass a list of messages
response = llm.invoke([message])
print(response)

content="The image depicts a close-up of a person's face, focusing on their cheek and nose area. The individual has light skin with dark hair and appears to have acne on their cheek.\n\n*   **Face:** \n    *   The face is the main subject of the image.\n    *   It is a close-up shot, focusing on the cheek and nose area.\n*   **Skin:**\n    *   The skin tone is light.\n    *   There are several acne spots on the cheek.\n*   **Hair:**\n    *   The hair is dark in color.\n    *   A few strands of hair are visible on the ear and cheek.\n*   **Acne:**\n    *   There are multiple acne spots on the cheek.\n    *   The acne spots vary in size and color, ranging from small and light pink to larger and more red.\n\nThe image presents a detailed view of a person's face, highlighting their skin condition." additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 189, 'prompt_tokens': 1902, 'total_tokens': 2091, 'completion_time': 0.427252817, 'prompt_time': 0.0614544, 'queue_ti

# Speech to Text (Voice of Patient)

In [ ]:
!pip install SpeechRecognition pydub python-dotenv
!apt-get install ffmpeg

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 47.0 MB/s eta 0:00:00
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 41 not upgraded.


In [ ]:
# ================================
# Step 1 — Audio Recording
# ================================
import logging
import speech_recognition as sr
from pydub import AudioSegment
from io import BytesIO

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def record_audio(file_path, timeout=20, phrase_time_limit=None):
    recognizer = sr.Recognizer()

    try:
        with sr.Microphone() as source:
            logging.info("Adjusting for ambient noise...")
            recognizer.adjust_for_ambient_noise(source, duration=1)
            logging.info("Start speaking now...")

            audio_data = recognizer.listen(source, timeout=timeout, phrase_time_limit=phrase_time_limit)
            logging.info("Recording complete.")

            wav_data = audio_data.get_wav_data()
            audio_segment = AudioSegment.from_wav(BytesIO(wav_data))
            audio_segment.export(file_path, format="mp3", bitrate="128k")

            logging.info(f"Audio saved to {file_path}")

    except Exception as e:
        logging.error(f"An error occurred: {e}")


audio_filepath = "/content/patient_voice_test.mp3"
# record_audio(audio_filepath)


# ================================
# Step 2 — Whisper STT (Raw Groq)
# ================================
import os
from groq import Groq

GROQ_API_KEY = os.environ.get("GROQ_API_KEY")
stt_model = "whisper-large-v3"

def transcribe_with_groq(stt_model, audio_filepath, GROQ_API_KEY):
    client = Groq(api_key=GROQ_API_KEY)

    with open(audio_filepath, "rb") as audio_file:
        transcription = client.audio.transcriptions.create(
            model=stt_model,
            file=audio_file,
            language="en"
        )

    return transcription.text



# =====================================================
# Step 3 — LangChain LLM (Processing the transcription)
# =====================================================
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage

# Initialize LangChain text model
text_llm = ChatGroq(
    groq_api_key=GROQ_API_KEY,
    model_name="llama-3.3-70b-versatile",
    temperature=0
)

def process_transcription_with_langchain(text):
    msg = HumanMessage(content=text)
    response = text_llm.invoke([msg])
    return response.content


# ===================
# RUN THE FULL PIPELINE
# ===================
if __name__ == "__main__":
    print("🎤 Transcribing audio with Whisper...")
    transcription = transcribe_with_groq(stt_model, audio_filepath, GROQ_API_KEY)
    print("TRANSCRIPTION:", transcription)

    print("\n🤖 Processing with LangChain LLM...")
    processed = process_transcription_with_langchain(transcription)
    print("\nMODEL RESPONSE:\n", processed)


🎤 Transcribing audio with Whisper...
TRANSCRIPTION:  Hello, my name is Hassan. How are you?

🤖 Processing with LangChain LLM...

MODEL RESPONSE:
 Hello Hassan, I'm doing well, thanks for asking. It's nice to meet you. I'm a large language model, so I don't have feelings or emotions like humans do, but I'm always happy to chat and help with any questions or topics you'd like to discuss. How about you, how's your day going so far?


# Step3 Text to Speech (Voice of Doctor)

In [ ]:
!pip install gTTS
!pip install elevenlabs
!pip install python-dotenv
!apt-get install ffmpeg -y

  Using cached elevenlabs-2.23.0-py3-none-any.whl.metadata (9.2 kB)
Using cached elevenlabs-2.23.0-py3-none-any.whl (1.1 MB)


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 41 not upgraded.


In [ ]:

# ================================
# IMPORTS
# ================================
import os
from gtts import gTTS
import elevenlabs
from elevenlabs.client import ElevenLabs
from IPython.display import Audio, display


# ================================
# HELPER — Play audio in Google Colab
# ================================
def play_audio_colab(filepath):
    display(Audio(filepath, autoplay=True))


# ================================
# STEP 1A — TEXT TO SPEECH USING gTTS
# ================================
def text_to_speech_with_gtts(input_text, output_filepath):
    audioobj = gTTS(
        text=input_text,
        lang="en",
        slow=False
    )
    audioobj.save(output_filepath)
    play_audio_colab(output_filepath)  # Play audio in Colab


# ================================
# STEP 1B — TEXT TO SPEECH USING ELEVENLABS
# ================================
ELEVENLABS_API_KEY = os.environ.get("ELEVEN_API_KEY")

def text_to_speech_with_elevenlabs(input_text, output_filepath):
    client = ElevenLabs(api_key=ELEVENLABS_API_KEY)

    audio = client.generate(
        text=input_text,
        voice="Aria",
        output_format="mp3_22050_32",
        model="eleven_turbo_v2"
    )

    elevenlabs.save(audio, output_filepath)
    play_audio_colab(output_filepath)  # Play audio in Colab


# ================================
# TESTING BOTH TTS MODELS
# ================================
print("Generating audio using gTTS...")
text_to_speech_with_gtts("Hi, this is AI with Hassan!", "gtts_testing.mp3")

print("Generating audio using ElevenLabs...")

text_to_speech_with_elevenlabs("Hello from ElevenLabs!", "eleven_testing.mp3")


Generating audio using gTTS...


Generating audio using ElevenLabs...


AttributeError: 'ElevenLabs' object has no attribute 'generate'